# Notebook 2 — Batch dataset generation

Scale Notebook 1 from *one seed × a few anomalies* to a **detection training set**:

`EDA → stratified real train/test → batch edit (load-once) → annotate → judge → export`

**Task:** object detection (Notebook 3 fine-tunes a pretrained YOLOv8n).
Rare classes stay rare in **real** train; synth bumps their share for the augmented train set.

Pick methods in [Notebook 1.5](01.5_method_comparison.ipynb); knobs below choose classes, counts, and method.

| Split | Role |
|-------|------|
| **Test (real only)** | Hold out enough rare images for credible AP |
| **Train real** | Remaining real images (rares stay rare) |
| **Train + synth** | Real train ∪ judge-accepted edits |


---
## 0. Setup

```bash
# From repo root:
uv sync --dev --group edge-case-image-generation
# Full toy extract (GPU/HF login; reads caps from data.yaml):
uv run python implementations/edge_case_image_generation/scripts/extract_mapillary_toy.py

# API key for the Vector proxy judge:
cp implementations/edge_case_image_generation/.env.example \
   implementations/edge_case_image_generation/.env
```

Judge defaults to **API vision chat** (`configs/default/judge.yaml`). For offline CPU workshops, override `judge.backend=qwen_vl` in `hardware/cpu.yaml` (no `.env` key needed).

Notebooks load `.env` on startup — no terminal `export` needed.

Select the project kernel, then run:


In [ ]:
import sys
from pathlib import Path


def _find_project_root() -> Path:
    here = Path.cwd().resolve()
    search = [here, *here.parents]
    for base in list(search):
        nested = base / "implementations" / "edge_case_image_generation"
        if nested.is_dir():
            search.append(nested)
    for base in search:
        if (base / "src" / "edgecase_synthesis").is_dir() and (base / "configs").is_dir():
            return base
    raise FileNotFoundError("Could not find edge_case_image_generation root")


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)
from edgecase_synthesis.config import load_env

load_env(PROJECT_ROOT)


---
## 1. Knobs

Hardcoding belongs **here**, not in package code. Change classes / counts / method freely.

Defaults target ~5% rare share in real train and ~20% after ~100 accepted synths per class
(with a full extract: ~320 scenes + 55 cone + 55 trash_bin; NB3 caps scene backgrounds at 150 for training).

**Pre-gen novelty:** each anomaly YAML defines `variations` + `prompt_template`. The batch
builds the cartesian product, shuffles once, then cycles — so short runs still spread across
axes (size/color/placement/…). Chosen combos are logged and written to the train manifest.


In [ ]:
import os

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

from edgecase_synthesis.config import load_config
from edgecase_synthesis.data import get_data_source_info, prepare_sample_images
from edgecase_synthesis.pipeline import resolve_method_map

# --- learner knobs ---------------------------------------------------------
DATASET = "mapillary_vistas"
HARDWARE = "gpu_l4"  # or "gpu_l4x2" (2× L4) / "cpu" for smoke run

# Which anomalies to synthesize + edit method (from Notebook 1.5).
METHOD_BY_ANOMALY = {
    "traffic_cone": "instruct",
    "trash_bin": "instruct",
}

# Real holdout for Notebook 3 eval (real images only).
TEST_COUNTS = {
    "scene": 90,
    "traffic_cone": 30,
    "trash_bin": 30,
}

# Disjoint clean scene seeds → synth (target accepts; stop early if hit).
N_SYNTH_SEEDS = {
    "traffic_cone": 100,
    "trash_bin": 100,
}
TARGET_ACCEPTS = {
    "traffic_cone": 100,
    "trash_bin": 100,
}

MAX_RETRIES = 2
SPLIT_SEED = 42
FOCUS_TAGS = ["scene", "traffic_cone", "trash_bin"]  # EDA bars / split keys
# ---------------------------------------------------------------------------

cfg = load_config(
    start=PROJECT_ROOT,
    overrides=[f"dataset_name={DATASET}", f"hardware={HARDWARE}"],
)
info = get_data_source_info(cfg)
prepare_sample_images(cfg=cfg)

samples_dir = Path(cfg.paths.samples_dir)
output_dir = Path(cfg.paths.outputs_dir) / "nb2"
output_dir.mkdir(parents=True, exist_ok=True)

# Prefer config stem_prefixes so multi-word tags (traffic_cone) resolve.
STEM_PREFIXES = list(cfg.data.get("stem_prefixes") or FOCUS_TAGS)

workshop = list(METHOD_BY_ANOMALY)
method_map = resolve_method_map(METHOD_BY_ANOMALY, workshop, cfg=cfg)

print(f"Dataset:  {cfg.dataset_name}")
print(f"Hardware: {cfg.hardware.name}  family={cfg.generation.family}")
print(f"Source:   {info.label} ({info.license})")
print(f"Samples:  {samples_dir}")
print(f"Export:   {output_dir}")
print("Method plan:")
for aid, method in method_map.items():
    print(f"  {aid:16s} → {method}")

---
## 2. EDA — class distribution (before)

Filename stem prefixes (`scene_`, `traffic_cone_`, …) are the extract buckets.
If counts are far below `TEST_COUNTS` / extract caps, re-run:

```bash
uv run python implementations/edge_case_image_generation/scripts/extract_mapillary_toy.py
```

Caps live in `configs/datasets/<dataset>/data.yaml` (`extract_max_generic`, `extract_max_per_class`).

In [ ]:
from edgecase_synthesis.eda import (
    clamp_counts,
    class_image_counts,
    group_by_tag,
    list_tagged_images,
    load_labels_for_dir,
    plot_class_bars,
    summarize_distribution,
)

all_paths = list_tagged_images(samples_dir)
by_tag = group_by_tag(all_paths, tags=FOCUS_TAGS, prefixes=STEM_PREFIXES)
labels = load_labels_for_dir(samples_dir)

before = summarize_distribution(by_tag, focus_tags=FOCUS_TAGS)
print("Image counts (before split/synth):")
for tag, n in before["counts"].items():
    share = before["shares"][tag]
    print(f"  {tag:16s}  {n:4d}  ({100 * share:5.1f}%)")
print(f"  total={before['total']}")

clean_counts = class_image_counts(by_tag, labels=labels, rare_classes=workshop)
if "clean_scene" in clean_counts:
    print(f"  clean_scene (no rare GT boxes): {clean_counts['clean_scene']}")

plot_class_bars(before["counts"], title="Before — extract buckets");

---
## 3. Stratified real train / test

Test is **real only** (no synth leakage). Counts auto-clamp if the local extract is still small.

In [ ]:
from edgecase_synthesis.eda import flatten_tag_groups, stratified_holdout, write_json

available = {t: len(by_tag.get(t) or []) for t in FOCUS_TAGS}
# Keep at least one image per tag in train when possible.
test_counts = clamp_counts(TEST_COUNTS, available, min_remaining=1)
if test_counts != TEST_COUNTS:
    print("Clamped TEST_COUNTS to available extract:")
    print(f"  requested={TEST_COUNTS}")
    print(f"  using    ={test_counts}")

train_real, test = stratified_holdout(by_tag, test_counts, seed=SPLIT_SEED)

train_summary = summarize_distribution(train_real, focus_tags=FOCUS_TAGS)
test_summary = summarize_distribution(test, focus_tags=FOCUS_TAGS)

print("Train (real):")
for tag, n in train_summary["counts"].items():
    print(f"  {tag:16s}  {n:4d}  ({100 * train_summary['shares'][tag]:5.1f}%)")
print("Test (real):")
for tag, n in test_summary["counts"].items():
    print(f"  {tag:16s}  {n:4d}  ({100 * test_summary['shares'][tag]:5.1f}%)")

write_json(
    output_dir / "split_summary.json",
    {
        "test_counts": test_counts,
        "train_real": train_summary,
        "test": test_summary,
        "seed": SPLIT_SEED,
    },
)
plot_class_bars(train_summary["counts"], title="Train real — class share");

---
## 4. Synth seeds (disjoint from train rare images)

Seeds are **train `scene_*`** images (prefer no rare GT boxes). They are *not* taken from the rare-class real pools, so synth does not duplicate test or rare train photos.

In [ ]:
from edgecase_synthesis.eda import allocate_budget, pick_synth_seeds

scene_train = list(train_real.get("scene") or [])
seed_request = allocate_budget(N_SYNTH_SEEDS, len(scene_train))
if seed_request != N_SYNTH_SEEDS:
    print("Clamped N_SYNTH_SEEDS to available train scenes:")
    print(f"  requested={N_SYNTH_SEEDS}")
    print(f"  using    ={seed_request}")

seeds_by_anomaly = pick_synth_seeds(
    scene_train,
    seed_request,
    labels=labels,
    rare_classes=workshop,
    seed=SPLIT_SEED,
)
for aid, paths in seeds_by_anomaly.items():
    print(f"  {aid:16s}  {len(paths)} scene seeds")

target_accepts = {
    k: min(int(TARGET_ACCEPTS.get(k, len(v))), len(v))
    for k, v in seeds_by_anomaly.items()
}
print("Target accepts:", target_accepts)

---
## 5. Batch synthesize (load models once)

Edit stack stays loaded for all first-pass generations; then we switch to the VLM judge.
Retries reload the edit stack only for failed items.

**Box gate (required for detection export):**
1. YOLO-World tries to box the rare object.
2. If it misses, we fall back to a box from the **edit mask** (where we painted).
3. Samples with **zero target boxes** cannot be accepted — retry/reject.

> On CPU this is slow — use `HARDWARE="gpu_l4"` for the full 100-seed run, or shrink `N_SYNTH_SEEDS` for a smoke test.

In [ ]:
from edgecase_synthesis.batch_runner import run_batch_synthesis

synth_dir = output_dir / "synthetic"
batch = run_batch_synthesis(
    seeds_by_anomaly,
    method_map,
    cfg=cfg,
    project_root=PROJECT_ROOT,
    synth_dir=synth_dir,
    max_retries=MAX_RETRIES,
    target_accepts=target_accepts,
    require_target_boxes=True,  # never export unboxed "accepted" synth
)

print("\nAcceptance rate per class:")
for aid, st in batch.stats.items():
    print(
        f"  {aid:16s}  accept={st.accepts}/{st.attempts}  "
        f"rate={100 * st.acceptance_rate:5.1f}%  "
        f"reject={st.rejects}  retry_events={st.retries}"
    )
empty = [r for r in batch.rejected if not r.get("has_target_boxes", True)]
if empty:
    print(f"Rejected for missing target boxes: {len(empty)}")

---
## 6. EDA — class distribution (after synth)

Compare **train real** vs **train real + accepted synth** (detection image counts per rare class).

In [ ]:
import matplotlib.pyplot as plt

# Image-level counts for rare classes (+ scene for context).
after_counts = dict(train_summary["counts"])
for sample in batch.accepted:
    after_counts[sample.anomaly_id] = after_counts.get(sample.anomaly_id, 0) + 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_class_bars(train_summary["counts"], title="Train real", ax=axes[0])
plot_class_bars(after_counts, title="Train real + accepted synth", ax=axes[1])
plt.show()

print("Rare-class share (images / train total):")
for split_name, counts in (("real", train_summary["counts"]), ("real+synth", after_counts)):
    total = sum(counts.values()) or 1
    print(f"  {split_name}  n={total}")
    for cls in workshop:
        n = counts.get(cls, 0)
        print(f"    {cls:16s}  {n:4d}  ({100 * n / total:5.1f}%)")

---
## 7. Export manifests for Notebook 3

Writes under `outputs/<dataset>/nb2/`:

- `synthetic/` — accepted RGB edits
- `labels_synthetic.json` — YOLO-World boxes for synth images
- `train_manifest.json` / `test_manifest.json`
- `run_stats.json` — acceptance rates + config snapshot

In [ ]:
from omegaconf import OmegaConf

from edgecase_synthesis.batch_export import export_nb2_dataset

config_snapshot = {
    "dataset": DATASET,
    "hardware": HARDWARE,
    "method_by_anomaly": dict(method_map),
    "test_counts": test_counts,
    "n_synth_seeds": seed_request,
    "target_accepts": target_accepts,
    "max_retries": MAX_RETRIES,
    "split_seed": SPLIT_SEED,
    "judge_model": str(cfg.judge.model_id),
    "judge_threshold": float(cfg.judge.threshold),
    "generation_family": str(cfg.generation.family),
}

paths = export_nb2_dataset(
    output_dir=output_dir,
    accepted=batch.accepted,
    train_real=train_real,
    test=test,
    real_labels=labels,
    run_stats=batch.stats,
    config_snapshot=config_snapshot,
)

print("Wrote:")
for key, path in paths.items():
    print(f"  {key:16s}  {path}")
print(f"\nAccepted synth images: {len(batch.accepted)}")
print("Next: Notebook 3 — fine-tune YOLOv8n on real vs real+synth.")

---
## Wrap-up

| Notebook | Role |
|----------|------|
| **1.5** | Compare methods; decide `METHOD_BY_ANOMALY` |
| **1** | Single-image loop + judge retry |
| **2** (this) | Batch EDA → synth → export |
| **3** | [Train detector: real vs real+synth](03_training_and_evaluation.ipynb) |

Fog / weather stays out of this detection story (no Mapillary fog class for AP).
Use Notebook 1 if you still want to demo diffuse effects + manual labels.
